# Expense RAG Prototype

A minimal RAG pipeline for the Personal Expense Intelligence Agent, using minsearch,
following the same structure used in the course: a `search` function, a `build_prompt`
function, an `llm` function, and a top-level `rag` function that chains them together.

This week's prototype uses a small synthetic dataset of Australian household expense
records (`data/expenses.json`) instead of real receipt images or OCR, so we can test
retrieval and RAG end to end before adding document extraction.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Synthetic household expense records for August and September 2026, covering
groceries, utilities, transport, dining, household, health, and subscriptions.
September includes a few deliberately unusual large expenses (a flight, a
replacement washing machine, a bulk grocery run, extra dining out) so the
"why were my expenses unusually high in September?" query has clear, traceable
evidence in the data.

Each record has: `id`, `date`, `merchant`, `category`, `description`, `total`.

In [2]:
with open("../data/expenses.json") as f:
    documents = json.load(f)

len(documents)

25

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

`merchant`, `category`, and `description` are indexed as free text, so a query
can match on the category name itself (e.g. "transport"), the merchant, or the
description. `category` is also indexed as a keyword field so it could be used
for exact-match filtering later (e.g. `filter_dict={"category": "household"}`).
`stop_words="english"` keeps common query words like "why" or "were" from
diluting the match on the words that actually matter, like "September".

In [3]:
index = Index(
    text_fields=["merchant", "category", "description"],
    keyword_fields=["category"],
    vectorizer_params={"stop_words": "english"},
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=13)

In [5]:
instructions = """
You're a personal expense assistant. Answer the QUESTION using only the
expense records provided in the CONTEXT below - each record is a household
transaction with a merchant, category, description, date, and total.

When you answer:
- Identify the specific merchants, categories, dates, and amounts that support
  your answer.
- If the question asks "why" something changed (e.g. a spending increase),
  point to the specific transactions that explain it.
- If the CONTEXT doesn't contain enough information to answer confidently,
  say so clearly instead of guessing.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

Three representative queries against the synthetic expense data.

In [8]:
queries = [
    "Why were my expenses unusually high in September?",
    "What were my largest household expenses?",
    "Which transactions were related to transport?",
]

for q in queries:
    print(f"Q: {q}")
    print(rag(q))
    print("-" * 80)

Q: Why were my expenses unusually high in September?
Your expenses were unusually high in September due to several significant transactions across various categories. Here are the key transactions that contributed:

1. **Utilities:**
   - **Origin Energy**: September electricity bill - **$122.50** (09/06)
   - **Telstra**: September mobile and home internet plan - **$95.00** (09/01)
   - **Sydney Water**: September water bill - **$41.30** (09/08)

2. **Dining:**
   - **The Local Hotel**: September birthday dinner for two - **$92.50** (09/05)
   - **Guzman y Gomez**: Dinner with friends - **$48.70** (09/12)

3. **Transport:**
   - **Qantas**: Flight from Sydney to Melbourne - **$389.00** (09/11)
   - **Uber**: Rides to and from the airport - **$67.40** (09/20)

4. **Household Expenses:**
   - **Bunnings Warehouse**: Replacement washing machine (after the old one broke) - **$648.00** (09/21)
   - **Kmart**: New desk and chair for the home office - **$149.00** (09/22)

5. **Groceries:**
 

## 5. Week 2 — Turning This Into a Tool-Calling Agent

The Week 1 pipeline above always does the same thing: search, then answer.
This section turns it into an agent that *decides* which tool to call based
on the question - exact filtering/math for precise questions, fuzzy search
for open-ended ones, and a small set of extraction tools for turning an
uploaded receipt into a new stored record.

The existing `documents` list and `index` built above are reused as-is.

### Expenses tools: `search_expenses`, `query_expenses`, `find_recurring_charges`

`search_expenses` wraps the Week 1 `search()` unchanged. `query_expenses` and
`find_recurring_charges` are plain Python over the raw `documents` list, not
minsearch - minsearch only ranks text matches, it can't compute an exact sum
or detect repeated amounts.

In [9]:
def search_expenses(query):
    """Semantic/fuzzy search over merchant, category, and description.
    Use for open-ended or explanatory questions."""
    # num_results is intentionally not model-controlled: a too-small value
    # (the model tried 5) silently drops the biggest-amount records, which
    # are usually exactly the evidence an explanatory question needs.
    return index.search(query, num_results=13)


def query_expenses(filters=None, group_by=None, aggregate=None):
    """Exact filter and/or aggregation over the expense table.
    Use when the answer needs a real number or a complete, deterministic match."""
    filters = filters or {}
    merchant = filters.get("merchant")
    category = filters.get("category")
    date_from = filters.get("date_from")
    date_to = filters.get("date_to")

    def matches(r):
        if merchant and merchant.lower() not in r["merchant"].lower():
            return False
        if category and r["category"].lower() != category.lower():
            return False
        if date_from and r["date"] < date_from:
            return False
        if date_to and r["date"] > date_to:
            return False
        return True

    rows = [r for r in documents if matches(r)]

    if not group_by and not aggregate:
        return rows

    def group_key(r):
        if group_by == "category":
            return r["category"]
        if group_by == "merchant":
            return r["merchant"]
        if group_by == "month":
            return r["date"][:7]
        return "all"

    groups = {}
    for r in rows:
        groups.setdefault(group_key(r), []).append(r["total"])

    def apply_aggregate(values):
        if aggregate == "count":
            return len(values)
        if aggregate == "avg":
            return round(sum(values) / len(values), 2)
        return round(sum(values), 2)  # default/"sum"

    return [{"group": k, "value": apply_aggregate(v)} for k, v in sorted(groups.items())]


def find_recurring_charges(min_occurrences=2, amount_tolerance_pct=10, category=None):
    """Groups transactions by merchant and flags ones that repeat at a similar
    amount, to catch recurring bills/subscriptions even outside the
    'subscriptions' category."""
    by_merchant = {}
    for r in documents:
        if category and r["category"].lower() != category.lower():
            continue
        by_merchant.setdefault(r["merchant"], []).append(r)

    results = []
    for merchant, txns in by_merchant.items():
        if len(txns) < min_occurrences:
            continue
        avg = sum(t["total"] for t in txns) / len(txns)
        tolerance = avg * (amount_tolerance_pct / 100)
        close = sorted(
            (t for t in txns if abs(t["total"] - avg) <= tolerance),
            key=lambda t: t["date"],
        )
        if len(close) < min_occurrences:
            continue
        results.append({
            "merchant": merchant,
            "category": close[0]["category"],
            "occurrences": [{"date": t["date"], "total": t["total"]} for t in close],
            "avg_amount": round(sum(t["total"] for t in close) / len(close), 2),
            "total_amount": round(sum(t["total"] for t in close), 2),
        })
    return results

### Extraction tools: `read_uploaded_document`, `check_duplicate_expense`, `write_expense_record`

`read_uploaded_document` reads from `data/uploads/` (seeded with one sample
receipt, `upload-001.txt`, for the demo below). Field extraction itself
(merchant, date, category, description, total) is done by the LLM's own
reasoning when it reads the document content - it is deliberately **not** a
tool, per the design.

In [10]:
from pathlib import Path
from datetime import datetime

UPLOADS_DIR = Path("../data/uploads")
EXPENSES_PATH = Path("../data/expenses.json")


def read_uploaded_document(document_id):
    """Retrieves the raw content of an uploaded receipt/bill/invoice so the
    LLM can read and extract fields from it directly."""
    path = UPLOADS_DIR / f"{document_id}.txt"
    if not path.exists():
        return {"error": f"document '{document_id}' not found"}

    stat = path.stat()
    return {
        "content": path.read_text(),
        "filename": path.name,
        "mime_type": "text/plain",
        "uploaded_at": datetime.fromtimestamp(stat.st_mtime).isoformat(),
    }


def check_duplicate_expense(merchant, date, total):
    """Checks whether a near-identical expense already exists, to avoid
    double-counting a receipt that was uploaded more than once."""
    for r in documents:
        if (
            r["merchant"].lower() == merchant.lower()
            and r["date"] == date
            and abs(r["total"] - total) < 0.01
        ):
            return {"is_duplicate": True, "existing_record": r}
    return {"is_duplicate": False}


def write_expense_record(id, date, merchant, category, description, total):
    """Persists a structured expense record extracted from a document.
    Stores to data/expenses.json for now; may move to Postgres later."""
    existing_ids = {r["id"] for r in documents}
    if id in existing_ids:
        id = f"{id}-{len(documents) + 1}"

    record = {
        "id": id,
        "date": date,
        "merchant": merchant,
        "category": category,
        "description": description,
        "total": total,
    }
    documents.append(record)
    EXPENSES_PATH.write_text(json.dumps(documents, indent=2))
    index.fit(documents)  # rebuild so the new record is searchable immediately

    return {"id": id, "status": "stored"}

### Tool schemas and dispatch table

The schemas are what the LLM sees to decide which tool to call. The dispatch
dict is how we actually invoke the right Python function once it decides.

In [11]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "name": "search_expenses",
        "description": (
            "Semantic search over expense merchant/category/description. Use "
            "for open-ended or explanatory questions (e.g. 'why was spending "
            "high in September?')."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
            },
            "required": ["query"],
        },
    },
    {
        "type": "function",
        "name": "query_expenses",
        "description": (
            "Exact filtering and/or aggregation over the expense table. Use "
            "when the answer needs a real number or a complete, deterministic "
            "match (totals, comparisons, exact merchant/date checks)."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "filters": {
                    "type": "object",
                    "properties": {
                        "merchant": {"type": "string"},
                        "category": {"type": "string"},
                        "date_from": {"type": "string", "description": "YYYY-MM-DD"},
                        "date_to": {"type": "string", "description": "YYYY-MM-DD"},
                    },
                },
                "group_by": {"type": "string", "enum": ["category", "month", "merchant"]},
                "aggregate": {"type": "string", "enum": ["sum", "count", "avg"]},
            },
        },
    },
    {
        "type": "function",
        "name": "find_recurring_charges",
        "description": (
            "Groups transactions by merchant across months and flags ones "
            "repeating at a similar amount. Use for subscriptions and "
            "recurring-bill questions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "min_occurrences": {"type": "integer", "default": 2},
                "amount_tolerance_pct": {"type": "number", "default": 10},
                "category": {"type": "string"},
            },
        },
    },
    {
        "type": "function",
        "name": "read_uploaded_document",
        "description": (
            "Retrieves the raw content of an uploaded receipt/bill/invoice by "
            "document id, so it can be read and analysed."
        ),
        "parameters": {
            "type": "object",
            "properties": {"document_id": {"type": "string"}},
            "required": ["document_id"],
        },
    },
    {
        "type": "function",
        "name": "check_duplicate_expense",
        "description": (
            "Checks whether a near-identical expense already exists (same "
            "merchant, date, and total). Call before writing a newly "
            "extracted expense."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "merchant": {"type": "string"},
                "date": {"type": "string", "description": "YYYY-MM-DD"},
                "total": {"type": "number"},
            },
            "required": ["merchant", "date", "total"],
        },
    },
    {
        "type": "function",
        "name": "write_expense_record",
        "description": (
            "Persists a structured expense record. Call only after the "
            "record has been extracted from a document and confirmed not to "
            "be a duplicate."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "id": {"type": "string"},
                "date": {"type": "string", "description": "YYYY-MM-DD"},
                "merchant": {"type": "string"},
                "category": {"type": "string"},
                "description": {"type": "string"},
                "total": {"type": "number"},
            },
            "required": ["id", "date", "merchant", "category", "description", "total"],
        },
    },
]

TOOL_FUNCTIONS = {
    "search_expenses": search_expenses,
    "query_expenses": query_expenses,
    "find_recurring_charges": find_recurring_charges,
    "read_uploaded_document": read_uploaded_document,
    "check_duplicate_expense": check_duplicate_expense,
    "write_expense_record": write_expense_record,
}

### Agent instructions

Specific, not generic - these are the rules the agent uses to pick the right
tool for each question, and to handle the extraction flow safely (read →
extract via its own reasoning → check for duplicates → write).

In [12]:
AGENT_INSTRUCTIONS = """
You're a personal expense agent for an Australian household. You have two
jobs, and you must choose the right tools for whichever one the user is
asking about:

1. ANSWERING QUESTIONS about existing spending (merchant, category,
   description, date, total).
2. PROCESSING AN UPLOADED DOCUMENT (a receipt, bill, or invoice) into a new
   structured expense record.

All expense records are dated 2026. When a question names a month without a
year (e.g. "September"), use that month in 2026 for `date_from`/`date_to` -
do not assume any other year.

Tool selection rules - follow these exactly:
- Use `search_expenses` for open-ended or explanatory questions where the
  wording matters more than an exact number (e.g. "why was spending high in
  September?", "what did I buy at Bunnings?").
- Use `query_expenses` whenever the answer needs an exact calculation or a
  complete, deterministic match: totals, month-over-month comparisons,
  category breakdowns, or checking for a specific charge.
- Use `find_recurring_charges` for questions about subscriptions or
  repeated/recurring bills, including ones that might not be labelled
  "subscriptions". Call it once without a `category` filter first - it
  already checks every merchant - and only narrow by category if the user
  asked about a specific one.
- Use `read_uploaded_document` whenever the user refers to an uploaded
  receipt, bill, or invoice (by its document id) that needs to be read and
  analysed.
- When processing an uploaded document: read it with `read_uploaded_document`,
  then do the field extraction yourself as part of your reasoning - identify
  the merchant, date (YYYY-MM-DD), category (one of: groceries, utilities,
  transport, dining, household, health, subscriptions), a short description,
  and the total. Do NOT call a separate tool to do the extraction - that is
  your job as the model.
- Before writing a newly extracted expense, call `check_duplicate_expense`
  with the merchant, date, and total. Only call `write_expense_record` if it
  is not a duplicate; if it is, tell the user instead of writing it again.
- Call `write_expense_record` only after the record has been extracted and
  checked - never call it speculatively or before you have real values for
  every field.

General rules:
- Ground every answer in the data returned by your tools. Cite the specific
  merchants, dates, and amounts that support your answer.
- If a tool's results don't contain enough information to answer
  confidently, say so clearly instead of guessing.
- When you've successfully stored a new expense record, confirm back to the
  user what was stored (merchant, date, category, total) and its id.
""".strip()

### The agent loop

A manual tool-calling loop against the OpenAI Responses API: call the model,
execute any `function_call` items via `TOOL_FUNCTIONS`, feed the results back
as `function_call_output` items, and repeat until the model returns plain
text. Each tool call is printed as it happens.

In [13]:
def run_agent(question, max_iterations=8, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": AGENT_INSTRUCTIONS},
        {"role": "user", "content": question},
    ]

    for _ in range(max_iterations):
        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=TOOL_SCHEMAS,
        )

        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        for call in function_calls:
            args = json.loads(call.arguments)
            print(f"TOOL CALL: {call.name}({args})")

            result = TOOL_FUNCTIONS[call.name](**args)

            messages.append({
                "type": "function_call",
                "call_id": call.call_id,
                "name": call.name,
                "arguments": call.arguments,
            })
            messages.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, default=str),
            })

    return "Reached the maximum number of tool calls without a final answer."

## 6. Try the Agent

### 6a. Expenses / Q&A flow

The same three questions from Week 1 - now watch the tool calls the agent
chooses for each one.

In [14]:
qa_questions = [
    "Why were my expenses unusually high in September?",
    "What were my largest household expenses?",
    "Which transactions were related to transport?",
    "Give me my total spending in September broken down by category.",
    "Do I have any recurring subscriptions or repeated charges?",
]

for q in qa_questions:
    print(f"Q: {q}")
    print(run_agent(q))
    print("-" * 80)

Q: Why were my expenses unusually high in September?
TOOL CALL: search_expenses({'query': 'high expenses in September 2026'})
Your expenses were unusually high in September primarily due to several significant charges across different categories:

1. **Transportation:**
   - **Qantas:** A flight from Sydney to Melbourne for a family visit cost **$389.00**.
   - **Uber:** Rides to and from the airport added **$67.40**.

2. **Utilities:**
   - **Origin Energy:** Electricity bill was **$122.50**.
   - **Telstra:** Mobile and home internet plan totaled **$95.00**.
   - **Sydney Water:** Water bill amounted to **$41.30**.

3. **Dining:**
   - **The Local Hotel:** A birthday dinner for two was **$92.50**.
   - **Guzman y Gomez:** Dinner with friends cost **$48.70**.

4. **Household:**
   - **Bunnings Warehouse:** Replacement washing machine was **$648.00**.
   - **Kmart:** New desk and chair for the home office totaled **$149.00**.

5. **Groceries:**
   - **Woolworths:** Grocery shopping exp

### 6b. Extraction flow

`data/uploads/upload-001.txt` is a sample Woolworths receipt (not already in
`data/expenses.json`). Expect: `read_uploaded_document` → the model extracts
the fields itself → `check_duplicate_expense` → `write_expense_record`.

In [15]:
print(run_agent(
    "I just uploaded a receipt, document id upload-001. Please read it, "
    "work out the expense details, and store it if it's not a duplicate."
))

TOOL CALL: read_uploaded_document({'document_id': 'upload-001'})
TOOL CALL: check_duplicate_expense({'merchant': 'Woolworths Metro', 'date': '2026-10-03', 'total': 31.5})
I've read the receipt from Woolworths Metro. Here are the details:

- **Merchant:** Woolworths Metro
- **Date:** 2026-10-03
- **Total:** $31.50
- **Category:** Groceries
- **Description:** Groceries including milk, bread, eggs, bananas, and chicken.

However, this expense is already recorded in our system as a duplicate. The existing record has the same date, merchant, and total.


## 7. Three Queries for Submission

Three fresh queries covering different tool paths: one that should hit
`search_expenses` (the main tool), one that should hit a different tool
(`query_expenses`), and one edge case outside the agent's current scope
(forecasting - explicitly deferred, see the tool design discussion).

In [16]:
submission_queries = [
    "Tell me about my spending at Bunnings Warehouse.",
    "What's the difference between my total spending in August 2026 and September 2026?",
    "How much do you predict I'll spend on groceries next month?",
]

for q in submission_queries:
    print(f"Q: {q}")
    print(run_agent(q))
    print("=" * 80)

Q: Tell me about my spending at Bunnings Warehouse.
TOOL CALL: search_expenses({'query': 'spending at Bunnings Warehouse'})
You have one expense recorded at Bunnings Warehouse:

- **Date:** September 14, 2026
- **Category:** Household
- **Description:** Replacement washing machine bought in September after the old one broke
- **Total:** $648.00

If you need more details or information on other merchants, just let me know!
Q: What's the difference between my total spending in August 2026 and September 2026?
TOOL CALL: query_expenses({'filters': {'merchant': '', 'category': '', 'date_from': '2026-08-01', 'date_to': '2026-08-31'}, 'group_by': 'month', 'aggregate': 'sum'})
TOOL CALL: query_expenses({'filters': {'merchant': '', 'category': '', 'date_from': '2026-09-01', 'date_to': '2026-09-30'}, 'group_by': 'month', 'aggregate': 'sum'})
In August 2026, your total spending was **$644.54**, while in September 2026, it increased significantly to **$1,976.39**. 

The difference in total spendin